# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import numpy as np


In [5]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
key_fields = ["impressions_window", "clicks_window", "click_through_rate", "content_age_days", "momentum"]
print(data_model[key_fields].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]))

       impressions_window  clicks_window  click_through_rate  \
count        1.833450e+05  183345.000000       183345.000000   
mean         4.084572e+03      12.215157            0.003724   
std          1.385366e+04      69.669931            0.020941   
min          1.000000e+00       0.000000            0.000000   
1%           1.000000e+00       0.000000            0.000000   
25%          7.500000e+01       0.000000            0.000000   
50%          4.760000e+02       1.000000            0.000359   
75%          2.622000e+03       5.000000            0.002801   
95%          1.965100e+04      56.000000            0.011442   
99%          5.513072e+04     187.000000            0.050000   
max          1.511334e+06   14225.000000            1.000000   

       content_age_days       momentum  
count     183345.000000  183345.000000  
mean         187.755390      -0.276047  
std          134.105123       1.689756  
min            0.000000      -1.000000  
1%             5.000000   

In [7]:
# Check heavy-tailedness explicitly: compare mean vs median
for col in ["impressions_window", "clicks_window"]:
    print(f"{col}: mean={data_model[col].mean():.1f}, median={data_model[col].median():.1f}, "
          f"max={data_model[col].max():.1f}")

impressions_window: mean=4084.6, median=476.0, max=1511334.0
clicks_window: mean=12.2, median=1.0, max=14225.0


In [10]:
# Log-transform for any future correlation work
data_model["log_impressions"] = np.log1p(data_model["impressions_window"])
print(data_model["log_impressions"].describe())

count    183345.000000
mean          6.048546
std           2.438116
min           0.693147
25%           4.330733
50%           6.167516
75%           7.872074
max          14.228504
Name: log_impressions, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
# --- Test 1: Volume ("high-impression pages decline more") ---
volume_check = duckdb.sql("""
    SELECT
        CASE WHEN impressions_window < 100 THEN 'low'
             WHEN impressions_window < 1000 THEN 'medium'
             ELSE 'high' END AS bucket,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM data_model GROUP BY bucket ORDER BY bucket
""").df()
print("Test 1: Volume\n", volume_check)

Test 1: Volume
    bucket      n  pct_declined
0    high  70848      0.368098
1     low  51891      0.036827
2  medium  60606      0.132017


In [12]:
# --- Test 2: Staleness ("older pages decline more") ---
age_check = duckdb.sql("""
    SELECT
        CASE WHEN content_age_days < 90 THEN '<90d'
             WHEN content_age_days < 180 THEN '90-180d'
             ELSE '180d+' END AS bucket,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM data_model GROUP BY bucket ORDER BY bucket
""").df()
print("Test 2: Staleness\n", age_check)

Test 2: Staleness
     bucket      n  pct_declined
0    180d+  89939      0.206729
1  90-180d  32710      0.196454
2     <90d  60696      0.180770


In [13]:
# --- Test 3: CTR ("low-CTR pages decline more") ---
ctr_median = data_model["click_through_rate"].median()
ctr_check = duckdb.sql(f"""
    SELECT
        CASE WHEN click_through_rate < {ctr_median} THEN 'below_median' ELSE 'above_median' END AS bucket,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM data_model GROUP BY bucket
""").df()
print("Test 3: CTR\n", ctr_check)

Test 3: CTR
          bucket      n  pct_declined
0  below_median  91672      0.006163
1  above_median  91673      0.386439




# Test 1
 — Volume: "high-impression pages decline more." Verdict: CONFIRMED. Low (n=51,891): 3.7%. Medium (n=60,606): 13.2%. High (n=70,848): 36.8%. All buckets well above the ~50-row floor. In practice: high-visibility pages carry real, substantial decline risk — likely because they have more room to lose.

# Test 2
— Staleness: "older pages decline more." Verdict: FALSE/MIXED. <90d: 18.1%, 90-180d: 19.6%, 180d+: 20.7% (n in the tens of thousands per bucket, well above the floor). The differences are small and don't show the strong pattern expected. In practice: on this Feb-April window, page age alone isn't a reliable decline predictor — a genuine negative result.

# Test 3
 — CTR: "low-CTR pages decline more." Verdict: OPPOSITE. Below-median CTR (n=91,672): 0.6% declined. Above-median CTR (n=91,673): 38.6% declined. In practice: this is a floor effect — pages with almost no clicks can't decline further, so low CTR looks "safe" only because there's nothing left to lose, not because the page is healthy.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [14]:
# Volume is directly behind FlyRank's "quick-win" flag logic
quick_win_threshold = data_model["impressions_window"].quantile(0.67)
flag_test = duckdb.sql(f"""
    SELECT
        impressions_window >= {quick_win_threshold} AS flagged_high_volume,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM data_model GROUP BY flagged_high_volume
""").df()
print(flag_test)

   flagged_high_volume       n  pct_declined
0                False  122831      0.099958
1                 True   60514      0.391860


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should trust volume as the primary prioritization signal — high-visibility pages are measurably more likely to be declining, confirmed both in raw buckets and at the exact threshold FlyRank's own quick-win flag uses. Staleness, despite being intuitive, showed no reliable relationship on this dataset and should not be used alone. CTR must never be used in its raw form (low CTR ≠ safe) — its apparent relationship is a floor effect, not a real risk signal, and using it naively would systematically miss genuinely at-risk high-traffic pages while wasting review time on already-quiet ones.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.